# Fraud Risk Model

**Goal:** rank insurance claims by fraud risk so reviewers can check the riskiest ones first.
**Input:** `claims_clean.csv` (15,420 claims, about 6% fraud). **Output:** `claims_scored.csv` with an out-of-fold risk score and tier for every claim.

## 1. Features and train/test split

In [1]:
# Load cleaned data and choose model features

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)

df = pd.read_csv("../Data/claims_clean.csv")

drop_cols = [
    "PolicyNumber", "RepNumber",                         # identifiers, not real predictors
    "Year", "Month", "MonthClaimed", "WeekOfMonth",      # date columns: fraud labels look uneven by month
    "WeekOfMonthClaimed", "DayOfWeek", "DayOfWeekClaimed",
    "Age", "AgeMissing",                                 # AgeOfPolicyHolder already covers age
]

X = df.drop(columns=drop_cols + ["FraudFound_P"])
y = df["FraudFound_P"]

X["DriverRating"] = X["DriverRating"].astype(str)        # 1-4 is a category, not a measurement

print(X.shape)   # expect (15420, 22)

(15420, 22)


## 2. Baseline model and review-capacity metric

In [2]:
# 80% train, 20% test; stratify keeps the ~6% fraud rate in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train fraud rate:", round(y_train.mean() * 100, 2), "%")
print("Test fraud rate:", round(y_test.mean() * 100, 2), "%")

Train fraud rate: 5.98 %
Test fraud rate: 6.0 %


In [3]:
# Baseline: logistic regression, with class_weight to handle the 94/6 imbalance

cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print(len(cat_cols), "categorical columns; numeric:", num_cols)

prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

logit = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
logit.fit(X_train, y_train)

proba = logit.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC:", round(average_precision_score(y_test, proba), 3),
      "(random guessing would be about", round(y_test.mean(), 3), ")")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

21 categorical columns; numeric: ['Deductible']
ROC-AUC: 0.811
PR-AUC: 0.158 (random guessing would be about 0.06 )
[[1727 1172]
 [  12  173]]
              precision    recall  f1-score   support

           0      0.993     0.596     0.745      2899
           1      0.129     0.935     0.226       185

    accuracy                          0.616      3084
   macro avg      0.561     0.765     0.485      3084
weighted avg      0.941     0.616     0.714      3084



In [4]:
# How much fraud is caught if reviewers only check the riskiest X% of claims?

def capture_table(y_true, scores, label):
    d = (pd.DataFrame({"y": np.asarray(y_true), "s": scores})
           .sort_values("s", ascending=False).reset_index(drop=True))
    total = d["y"].sum()
    rows = []
    for pct in [5, 10, 20, 30, 50]:
        n = int(len(d) * pct / 100)
        caught = int(d["y"].iloc[:n].sum())
        rows.append({"model": label, "review_top_pct": pct, "claims_reviewed": n,
                     "fraud_caught": caught,
                     "share_of_fraud_caught_pct": round(100 * caught / total, 1),
                     "precision_pct": round(100 * caught / n, 1)})
    return pd.DataFrame(rows)

capture_table(y_test, proba, "Logistic")

,model,review_top_pct,claims_reviewed,fraud_caught,share_of_fraud_caught_pct,precision_pct
0,Logistic,5,154,27,14.6,17.5
1,Logistic,10,308,53,28.6,17.2
2,Logistic,20,616,99,53.5,16.1
3,Logistic,30,925,142,76.8,15.4
4,Logistic,50,1542,180,97.3,11.7


## 3. Comparing models

In [5]:
# Compare logistic regression with random forest and gradient boosting on the same test set

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
prep_dense = ColumnTransformer([("cat", ohe, cat_cols), ("num", "passthrough", num_cols)])

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1),
    "GradientBoosting": HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=300, random_state=42),
}

results = [capture_table(y_test, proba, "Logistic")]
for name, m in models.items():
    pipe = Pipeline([("prep", prep_dense), ("model", m)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    print(name, "ROC-AUC:", round(roc_auc_score(y_test, p), 3),
          "| PR-AUC:", round(average_precision_score(y_test, p), 3))
    results.append(capture_table(y_test, p, name))

pd.concat(results).query("review_top_pct in [10, 20, 30]")

RandomForest ROC-AUC: 0.816 | PR-AUC: 0.19
GradientBoosting ROC-AUC: 0.827 | PR-AUC: 0.22


,model,review_top_pct,claims_reviewed,fraud_caught,share_of_fraud_caught_pct,precision_pct
1,Logistic,10,308,53,28.6,17.2
2,Logistic,20,616,99,53.5,16.1
3,Logistic,30,925,142,76.8,15.4
1,RandomForest,10,308,57,30.8,18.5
2,RandomForest,20,616,100,54.1,16.2
3,RandomForest,30,925,137,74.1,14.8
1,GradientBoosting,10,308,61,33.0,19.8
2,GradientBoosting,20,616,110,59.5,17.9
3,GradientBoosting,30,925,140,75.7,15.1


## 4. Cross-validation and feature importance

In [6]:
# 5-fold stratified cross-validation: is the model ranking stable across splits?
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipes = {
    "Logistic": logit,
    "RandomForest": Pipeline([("prep", prep_dense), ("model", models["RandomForest"])]),
    "GradientBoosting": Pipeline([("prep", prep_dense), ("model", models["GradientBoosting"])]),
}

rows = []
for name, p in pipes.items():
    s = cross_validate(p, X, y, cv=cv, n_jobs=1,
                       scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"})
    rows.append({"model": name,
                 "ROC-AUC": f"{s['test_roc_auc'].mean():.3f} ± {s['test_roc_auc'].std():.3f}",
                 "PR-AUC": f"{s['test_pr_auc'].mean():.3f} ± {s['test_pr_auc'].std():.3f}"})
pd.DataFrame(rows)

,model,ROC-AUC,PR-AUC
0,Logistic,0.806 ± 0.009,0.164 ± 0.010
1,RandomForest,0.813 ± 0.008,0.185 ± 0.012
2,GradientBoosting,0.823 ± 0.005,0.229 ± 0.020


In [ ]:
# Permutation importance: how much does the model's PR-AUC drop when a column is shuffled?

from sklearn.inspection import permutation_importance

gb = Pipeline([("prep", prep_dense), ("model", models["GradientBoosting"])])
gb.fit(X_train, y_train)

perm = permutation_importance(gb, X_test, y_test, scoring="average_precision",
                              n_repeats=5, random_state=42, n_jobs=1)
imp = (pd.Series(perm.importances_mean, index=X_test.columns)
         .sort_values(ascending=False).head(10))
print(imp.round(4))

## 5. Follow-up: address change before the claim

In [ ]:
# Fraud rate by time between address change and claim
(df.groupby("AddressChange_Claim")["FraudFound_P"]
   .agg(claims="count", fraud_cases="sum", fraud_rate="mean")
   .assign(fraud_rate=lambda d: (d["fraud_rate"] * 100).round(2))
   .sort_values("fraud_rate", ascending=False))

In [ ]:
# Does the address-change signal hold inside the high-risk segment (at-fault, All Perils or Collision)?

seg = df[df["BasePolicy"].isin(["All Perils", "Collision"]) & (df["Fault"] == "Policy Holder")]
(seg.groupby("AddressChange_Claim")["FraudFound_P"]
    .agg(claims="count", fraud_cases="sum", fraud_rate="mean")
    .assign(fraud_rate=lambda d: (d["fraud_rate"] * 100).round(2)))

Claims filed 2 to 3 years after an address change show a 17.5% fraud rate (291 claims), but most of that fraud (36 of 51 cases) sits outside the high-risk segment. The group is small and the pattern is not monotonic, so this is a signal to monitor, not a rule.

## 6. Scoring every claim for the dashboard

In [ ]:
# Out-of-fold fraud-risk scores for every claim (for the dashboard)

from sklearn.model_selection import cross_val_predict

final = Pipeline([("prep", prep_dense), ("model", models["GradientBoosting"])])
oof = cross_val_predict(final, X, y, cv=cv, method="predict_proba")[:, 1]

scored = df.copy()
scored["risk_score"] = oof.round(4)
scored["risk_rank_pct"] = scored["risk_score"].rank(ascending=False, method="first") / len(scored)
scored["risk_tier"] = np.where(scored["risk_rank_pct"] <= 0.10, "High",
                        np.where(scored["risk_rank_pct"] <= 0.30, "Medium", "Low"))

summary = (scored.groupby("risk_tier")["FraudFound_P"]
                 .agg(claims="count", fraud_cases="sum", fraud_rate="mean"))
summary["fraud_rate"] = (summary["fraud_rate"] * 100).round(2)
summary["share_of_all_fraud_pct"] = (100 * summary["fraud_cases"] / scored["FraudFound_P"].sum()).round(1)
print(summary.loc[["High", "Medium", "Low"]])

scored.to_csv("../Data/claims_scored.csv", index=False)

## Modeling findings

- Fraud is rare (6%), so models are judged by PR-AUC, ROC-AUC, and the share of fraud caught when only the riskiest claims are reviewed, not accuracy.
- Gradient boosting performed best in 5-fold cross-validation (ROC-AUC 0.823, PR-AUC 0.229, versus 0.806 and 0.164 for logistic regression).
- Top drivers: fault, coverage type, address change before the claim, policy type, and policyholder age.
- Reviewing the riskiest 30% of claims (High + Medium tiers) catches about 73% of fraud; the High tier alone (10% of claims) catches 35% at a 21% fraud rate.
- Limitations: modest precision, labels reflect detected fraud only, 1994-1996 data, and date columns were excluded because fraud labels look uneven by month.

- Address change: claims filed 2-3 years after an address change had a 17.5% fraud rate (291 claims); most of that fraud falls outside the high-risk segment, so fault and coverage alone would miss it.